# Ingredient Decoder - LLaMA 3 8B Fine-Tuning (Stable 2026 Colab Version)

In [ ]:

# CLEAN INSTALL BLOCK
!pip uninstall -y trl unsloth unsloth-zoo xformers bitsandbytes -q
!pip install unsloth
!pip install trl==0.23.1
!pip install peft accelerate bitsandbytes datasets transformers

print("Installation complete. Now RESTART runtime before continuing.")


  Using cached unsloth-2026.2.1-py3-none-any.whl.metadata (69 kB)
  Using cached unsloth_zoo-2026.2.1-py3-none-any.whl.metadata (32 kB)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached unsloth-2026.2.1-py3-none-any.whl (432 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
Using cached unsloth_zoo-2026.2.1-py3-none-any.whl (376 kB)
Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl (3.3 MB)
  Using cached trl-0.23.1-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.23.1-py3-none-any.whl (564 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
Installation complete. Now RESTART runtime before continuing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create permanent project directory in Drive
!mkdir -p /content/drive/MyDrive/ingredient_decoder_project/data
!mkdir -p /content/drive/MyDrive/ingredient_decoder_project/checkpoints
!mkdir -p /content/drive/MyDrive/ingredient_decoder_project/final_model

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# OPTION 1: Copy from Google Drive (if already uploaded)
# !cp "/content/drive/MyDrive/ingredient_safety_unique.jsonl" /content/ingredient_decoder/data/

# OPTION 2: Upload directly via Colab file uploader
from google.colab import files
uploaded = files.upload()  # Select ingredient_safety_unique.jsonl from your local machine

# Move uploaded file to data directory
import shutil
for filename in uploaded.keys():
    shutil.move(filename, f"/content/ingredient_decoder/data/{filename}")

print(f"Uploaded files: {list(uploaded.keys())}")

Saving ingredient_safety_unique.jsonl to ingredient_safety_unique.jsonl
Uploaded files: ['ingredient_safety_unique.jsonl']


In [ ]:
!ls /content/ingredient_decoder/data

ingredient_safety_unique.jsonl


In [ ]:

from datasets import load_dataset
import json

dataset_path = "/content/ingredient_decoder/data/ingredient_safety_unique.jsonl"

dataset = load_dataset(
    "json",
    data_files=dataset_path,
    split="train"
)

print("Total samples:", len(dataset))
print("\nSample entry:")
print(json.dumps(dataset[0], indent=2))


Generating train split: 0 examples [00:00, ? examples/s]

Total samples: 6184

Sample entry:
{
  "instruction": "Analyze this food ingredient and classify its safety level as Safe, Moderate, or Harmful. Provide a brief explanation.",
  "input": "glucose syrup",
  "output": "Risk Level: Safe\n\nExplanation: This ingredient is generally recognized as safe for consumption. FDA GRAS (Generally Recognized As Safe) certified."
}


In [ ]:

from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Llama-3-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=False
)

print("Model loaded successfully!")
print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded successfully!
GPU Memory allocated: 5464.70 MB


In [ ]:

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=3407,
)

print("LoRA adapters added.")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.2.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA adapters added.


In [ ]:

def format_instruction(example):
    return {
        "text": f"""### Instruction:
Analyze the ingredient and classify it as Safe, Moderate, or Harmful.

### Input:
{example['input']}

### Output:
{example['output']}
"""
    }

dataset = dataset.map(format_instruction)
dataset = dataset.remove_columns([col for col in dataset.column_names if col != "text"])

print(dataset[0])


Map:   0%|          | 0/6184 [00:00<?, ? examples/s]

{'text': '### Instruction:\nAnalyze the ingredient and classify it as Safe, Moderate, or Harmful.\n\n### Input:\nglucose syrup\n\n### Output:\nRisk Level: Safe\n\nExplanation: This ingredient is generally recognized as safe for consumption. FDA GRAS (Generally Recognized As Safe) certified.\n'}


In [ ]:

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/6184 [00:00<?, ? examples/s]

In [ ]:

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ingredient_decoder_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_dataset,
    dataset_text_field="text",
    max_seq_length=256,
    args=training_args
)


In [ ]:

trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,184 | Num Epochs = 1 | Total steps = 3,092
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,12.585800
20,6.459600
30,5.381700
40,5.228400
50,5.199500
60,5.236800
70,5.249300
80,5.202200
90,5.222700
100,5.210900


TrainOutput(global_step=3092, training_loss=5.218704185695574, metrics={'train_runtime': 7409.3081, 'train_samples_per_second': 0.835, 'train_steps_per_second': 0.417, 'total_flos': 1.4336971754805658e+17, 'train_loss': 5.218704185695574, 'epoch': 1.0})

In [2]:
!ls /content/drive/MyDrive/ingredient_decoder_checkpoints

ls: cannot access '/content/drive/MyDrive/ingredient_decoder_checkpoints': No such file or directory


In [1]:
trainer.train(resume_from_checkpoint=True)

trainer.save_model("/content/drive/MyDrive/ingredient_decoder_project/final_model")
tokenizer.save_pretrained("/content/drive/MyDrive/ingredient_decoder_project/final_model")

print("Final model permanently saved to Drive.")

NameError: name 'trainer' is not defined